# TweetAge v8: PPM-C de caracteres por clase (Teahan y Harper, 2003)

**Objetivo.** Prueba de cierre de la familia generativa: clasificacion por compresion, con un modelo PPM por clase como en Teahan y Harper (2003), *Using compression-based language models for text categorization*. Se compara contra la referencia A en los mismos pliegues; el metodo hermano de esta ronda es Naive Bayes (TweetAge_v8_naive_bayes.ipynb).

**Metodo.** Un modelo PPM-C de caracteres por rango de edad, orden maximo 4, estatico (entrenado con los textos de la clase en el ajuste del pliegue). Para cada simbolo se parte del contexto mas largo disponible (hasta 4 caracteres) y se baja de orden escapando: en cada contexto con n simbolos distintos y total t, p(simbolo) = c / (t + n) y p(escape) = n / (t + n) (metodo C). Con exclusion: al bajar de orden se descartan los simbolos ya vistos en ordenes superiores. El orden -1 es uniforme sobre el alfabeto del ajuste (caracteres + fin de texto + desconocido) menos los excluidos. Los conteos se actualizan en todos los ordenes (sin exclusion de actualizacion). Puntuacion: log2 P(texto + fin) en bits + log2 prior. Implementado desde cero, sin librerias de compresion.

**Historial.** v6 (Kneser-Ney modificado, particion agrupada) es la version valida publicada: OOF sin calibrar 0,44106, calibrado 0,44353, Kaggle publico 0,45737. Las rondas v7 sobre la particion v7_lower (ordinal 0,42321; descuento con concentracion 0,440875; normalizacion A 0,440081; ocho configuraciones de hiperparametros sin superar 0,440081; modelo de lenguaje neuronal GRU con diferencia -0,029712 en el pliegue 1) no superaron la referencia. Referencia historica de v5: calibrado 0,43847, sobre una particion distinta.

**Referencia A** (ya medida en v7_lower, mismos 5 pliegues de StratifiedKFold(5, shuffle=True, random_state=42) sobre los grupos del desarrollo): 0,75 x KN caracteres(4) + 0,25 x KN palabras(3) + log prior; F1 OOF 0,440081; por pliegue 0,440843 / 0,432485 / 0,440697 / 0,447354 / 0,439014. Se recalcula aqui y se verifica con tolerancia 1e-6; si no coincide, la corrida se detiene.

**Regla de decision (fijada antes de ejecutar).** El metodo se acepta solo si la media de las cinco diferencias pareadas de F1 macro contra A es > 0 y mayor que su desviacion estandar muestral (ddof=1). Si no pasa: "NO PASA", sin holdout ni CSV. Si pasa: holdout evaluado una sola vez y CSV `age_submission_v8_ppm.csv` con escritura exclusiva. Heuristica, no prueba formal.


In [1]:
import ast
import csv
import gc
import hashlib
import html
import json
import math
import platform
import re
import time
from collections import Counter
from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from IPython.display import display, Markdown

SEED=42
np.random.seed(SEED)
CLASSES=np.array(['13-17','18-24','25-34','35-49','50-64','65-xx'])
BASE_EPSILON=1e-6
BOS,EOS,UNK='\x02','\x03','\x01'
CHAR_ORDERS=[4]
WORD_ORDERS=[3]
REF_CHAR_ORDER,REF_WORD_ORDER,REF_WEIGHT=4,3,0.75
REF_FOLD_F1=np.array([0.440843,0.432485,0.440697,0.447354,0.439014])
REF_OOF_F1=0.440081
REF_TOLERANCE=1e-6
METHOD='PPM'
METHOD_LABEL='PPM-C de caracteres, orden 4, con exclusion'
roots=[Path.cwd(),Path.cwd()/'workshops'/'Kaggle-tweet-age',Path.cwd()/'natural-language-processing-workshops'/'workshops'/'Kaggle-tweet-age']
OUT_DIR=next((candidate for candidate in roots if (candidate/'TweetAge_v8_ppm.ipynb').is_file()),None)
assert OUT_DIR is not None,'Ejecute desde la carpeta del notebook o la raiz del repo.'
NOTEBOOK_PATH=OUT_DIR/'TweetAge_v8_ppm.ipynb'
SUB_PATH=OUT_DIR/'age_submission_v8_ppm.csv'
SPLIT_PATH=OUT_DIR/'split_idx_v7_lower_seed42.npz'
DATA_DIR=OUT_DIR/'tweet-age-2026-puj'
assert not SUB_PATH.exists(),'El CSV ya existe; no se sobrescribe.'
INPUT_HASHES={name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in ['age_train.csv','age_test.csv','age_submission.csv']}
PROTECTED_HASHES={path.name:hashlib.sha256(path.read_bytes()).hexdigest() for path in OUT_DIR.iterdir() if path.is_file() and path!=NOTEBOOK_PATH}
assert PROTECTED_HASHES[SPLIT_PATH.name]=='692fcfd2fc5a8e1ead91763d4741da5d14f86fecd201b2e419c5e7d47c16be83','Hash del NPZ v7_lower distinto; detener.'
print('Python',platform.python_version(),'numpy',np.__version__,'pandas',pd.__version__,'sklearn',sklearn.__version__)
print('Metodo:',METHOD_LABEL,'| semilla',SEED,'| notebook',NOTEBOOK_PATH.name)
print('SHA-256 NPZ v7_lower:',PROTECTED_HASHES[SPLIT_PATH.name])
print('SHA-256 age_train.csv:',INPUT_HASHES['age_train.csv'])

def check_compliance(notebook):
    # Guardia AST de v6 mas prohibiciones de esta ronda: vectorizadores, Naive Bayes de sklearn, matrices dispersas, compresores externos, torch.
    forbidden = {'tfidfvectorizer', 'countvectorizer', 'hashingvectorizer', 'dictvectorizer', 'feature_extraction', 'linear_model', 'transformers', 'transformer', 'embedding', 'embeddings', 'word2vec', 'glove', 'gensim', 'pretrained', 'from_pretrained', 'multinomialnb', 'complementnb', 'bernoullinb', 'gaussiannb', 'categoricalnb', 'naive_bayes', 'sparse', 'csr_matrix', 'csc_matrix', 'coo_matrix', 'lil_matrix', 'zlib', 'bz2', 'lzma', 'gzip', 'zipfile', 'torch'}
    problems = []
    for cell_number, cell in enumerate(notebook['cells'], 1):
        if cell['cell_type'] != 'code':
            continue
        source = ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
        for node in ast.walk(ast.parse(source)):
            names = []
            if isinstance(node, ast.Name):
                names = [node.id]
            elif isinstance(node, ast.Attribute):
                names = [node.attr]
            elif isinstance(node, ast.Import):
                names = [alias.name for alias in node.names]
            elif isinstance(node, ast.ImportFrom):
                names = [node.module or ''] + [alias.name for alias in node.names]
            for name in names:
                if any(part.lower() in forbidden for part in name.split('.')):
                    problems.append((cell_number, node.lineno, name))
            if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in {'eval', 'exec', '__import__'}:
                problems.append((cell_number, node.lineno, 'ejecucion dinamica'))
    if problems:
        raise RuntimeError(f'Guardia AST: codigo prohibido {problems}')
    return True

with NOTEBOOK_PATH.open(encoding='utf-8') as handle:
    assert check_compliance(json.load(handle))
print('Guardia AST de v6 superada antes de entrenar; sin vectorizadores, NB de sklearn, matrices de conteos, compresores externos ni torch.')


Python 3.12.14 numpy 2.5.4 pandas 3.0.6 sklearn 1.9.1
Metodo: PPM-C de caracteres, orden 4, con exclusion | semilla 42 | notebook TweetAge_v8_ppm.ipynb
SHA-256 NPZ v7_lower: 692fcfd2fc5a8e1ead91763d4741da5d14f86fecd201b2e419c5e7d47c16be83
SHA-256 age_train.csv: 39c6c8270366fe155f3c8dd232e31867bee659af8908121d04b6bd89bd163748
Guardia AST de v6 superada antes de entrenar; sin vectorizadores, NB de sklearn, matrices de conteos, compresores externos ni torch.


## Datos, normalizacion A y verificacion del NPZ v7_lower
Celda identica a la de v7 hiperparametros: hashes del train y de los grupos, cero fuga en minusculas en holdout y en los cinco pliegues.

In [2]:
def verify_csv(path, columns, delimiter, bom):
    with path.open('rb') as handle:
        prefix = handle.read(32768)
    first_line = prefix.decode('utf-8-sig').splitlines()[0]
    actual = csv.Sniffer().sniff(first_line, delimiters=',;\t|').delimiter
    header = next(csv.reader([first_line], delimiter=actual))
    assert actual == delimiter and prefix.startswith(b'\xef\xbb\xbf') == bom, (path.name, actual)
    assert header == columns, (path.name, header)
    print(path.name, 'BOM=', bom, 'separador=', repr(actual), 'columnas=', header)
    return header

URL_RE = re.compile(r'https?://[^\s]+|www\.[^\s]+', re.IGNORECASE)

MENTION_RE = re.compile(r'@\w+', re.UNICODE)

REPEAT_RE = re.compile(r'([^\W\d_])\1{3,}', re.UNICODE)

def replace_url(match):
    value = match.group(0)
    core = value.rstrip('.,!?;:)]}')
    return '<url>' + value[len(core):]

def normalize_text(text):
    return REPEAT_RE.sub(lambda match: match.group(1) * 3, MENTION_RE.sub('<user>', URL_RE.sub(replace_url, text)))

verify_csv(DATA_DIR/'age_train.csv',['id','text','age_range'],';',True)
reference_columns=verify_csv(DATA_DIR/'age_submission.csv',['id','age_range'],',',False)
train_raw=pd.read_csv(DATA_DIR/'age_train.csv',sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
assert list(train_raw.columns)==['id','text','age_range']
clean=train_raw.text.fillna('').str.strip().map(html.unescape)
excluded=clean.eq('')|clean.str.contains('#\u00bfNOMBRE',regex=False)
train=train_raw.loc[~excluded].copy()
train['raw_row']=np.flatnonzero(~excluded.to_numpy())
train['text']=clean.loc[~excluded]
train=train.reset_index(drop=True)
assert train.id.is_unique and set(train.age_range)==set(CLASSES)
train['normalized']=train.text.map(normalize_text)
y_all=pd.Categorical(train.age_range,categories=CLASSES).codes.astype(np.int64)
with np.load(SPLIT_PATH,allow_pickle=False) as saved:
    split={name:saved[name].copy() for name in saved.files}
assert str(split['train_sha256'])==INPUT_HASHES['age_train.csv'] and int(split['seed'])==SEED
assert np.array_equal(split['retained_raw_idx'],train.raw_row.to_numpy())
group_keys=np.array([text.lower() for text in train.normalized],dtype=object)
groups_hash=hashlib.sha256(json.dumps(group_keys.tolist(),ensure_ascii=False,separators=(',',':')).encode('utf-8')).hexdigest()
assert groups_hash==str(split['groups_sha256'])
group_ids,unique_keys=pd.factorize(group_keys,sort=True)
assert np.array_equal(group_ids,split['group_ids'])
idx_development,idx_holdout=split['train_idx'],split['valid_idx']
dev_groups,holdout_groups=split['train_groups'],split['valid_groups']
assert np.array_equal(np.sort(np.concatenate([idx_development,idx_holdout])),np.arange(len(train)))
assert np.array_equal(train.raw_row.to_numpy()[idx_development],split['raw_train_idx'])
assert np.array_equal(train.raw_row.to_numpy()[idx_holdout],split['raw_valid_idx'])
assert set(group_ids[idx_development])==set(dev_groups) and set(group_ids[idx_holdout])==set(holdout_groups)
assert not np.intersect1d(dev_groups,holdout_groups).size
assert not set(group_keys[idx_development])&set(group_keys[idx_holdout])
group_counts=np.zeros((len(unique_keys),6),dtype=np.int64)
np.add.at(group_counts,(group_ids,y_all),1)
group_targets=group_counts.argmax(axis=1)
texts_development=train.normalized.iloc[idx_development].to_numpy()
y_development=y_all[idx_development]
groups_development=group_ids[idx_development]
group_folds=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(dev_groups,group_targets[dev_groups]))
folds=[]
leak_rows=[dict(validacion='holdout',textos_compartidos=0)]
for fold_number,(fit_positions,app_positions) in enumerate(group_folds,1):
    fit_idx=np.flatnonzero(np.isin(groups_development,dev_groups[fit_positions]))
    app_idx=np.flatnonzero(np.isin(groups_development,dev_groups[app_positions]))
    shared={text.lower() for text in texts_development[fit_idx]}&{text.lower() for text in texts_development[app_idx]}
    leak_rows.append(dict(validacion=str(fold_number),textos_compartidos=len(shared)))
    if shared:
        print('DETENER: fuga en pliegue',fold_number,'ejemplos',sorted(shared)[:10])
        raise RuntimeError('Fuga en minusculas; no entrenar.')
    folds.append((fit_idx,app_idx))
print('NPZ REUTILIZADO SIN REGENERAR:',SPLIT_PATH.name)
print('SHA-256 train y grupos correctos. SHA-256 grupos:',groups_hash)
print('Filas train original/limpio/excluidas:',len(train_raw),len(train),int(excluded.sum()))
print('Desarrollo grupos/filas:',len(dev_groups),len(idx_development),'holdout grupos/filas:',len(holdout_groups),len(idx_holdout))
print('Textos solo URL:',int(train.normalized.str.strip().eq('<url>').sum()),'solo usuario:',int(train.normalized.str.strip().eq('<user>').sum()))
display(pd.DataFrame(leak_rows))


age_train.csv BOM= True separador= ';' columnas= ['id', 'text', 'age_range']
age_submission.csv BOM= False separador= ',' columnas= ['id', 'age_range']


NPZ REUTILIZADO SIN REGENERAR: split_idx_v7_lower_seed42.npz
SHA-256 train y grupos correctos. SHA-256 grupos: f482fe9f4a7495a0050b857b8eda3b3bc9177c31e2ae6f10c9138cfe8df2b1a8
Filas train original/limpio/excluidas: 169336 169181 155
Desarrollo grupos/filas: 126309 132987 holdout grupos/filas: 31184 36194
Textos solo URL: 3785 solo usuario: 287


,validacion,textos_compartidos
0,holdout,0
1,1,0
2,2,0
3,3,0
4,4,0
5,5,0


## Kneser-Ney modificado de v6 (solo para recalcular la referencia A)

In [3]:
TOKEN_RE = re.compile(r'<url>|<user>|\w+|[^\w\s]', re.UNICODE)

def tokenize(text, level):
    return list(text) if level == 'char' else TOKEN_RE.findall(text.lower())

def make_encoder(texts, level):
    frequencies = Counter()
    for text in texts:
        frequencies.update(tokenize(text, level))
    cutoff = 1 if level == 'char' else 2
    return {token: chr(index + 256) for index, token in enumerate(sorted(token for token, count in frequencies.items() if count >= cutoff))}

def encode_text(text, level, mapping):
    return ''.join(mapping.get(token, UNK) for token in tokenize(text, level))

def estimate_discounts(counts):
    histogram = Counter(counts.values())
    count1, count2, count3, count4 = [histogram.get(value, 0) for value in [1, 2, 3, 4]]
    factor = count1 / (count1 + 2 * count2) if count1 + 2 * count2 else 0.0
    first = 1 - 2 * factor * count2 / count1 if count1 and count2 else 0.75
    second = 2 - 3 * factor * count3 / count2 if count1 and count2 else 0.75
    third = 3 - 4 * factor * count4 / count3 if count1 and count2 and count3 else 0.75
    values = tuple(float(min(bucket - 1e-6, max(1e-6, value))) for bucket, value in enumerate([first, second, third], 1))
    return values, dict(N1=count1, N2=count2, N3=count3, N4=count4)

def discount_for(count, discounts):
    return discounts[min(count, 3) - 1] if count > 0 else 0.0

class ModifiedKN:
    def __init__(self, max_order, support):
        self.max_order = max_order
        self.support = set(support) | {EOS, UNK}

    def fit(self, sequences):
        highest = max(2, self.max_order)
        self.raw = {order: Counter() for order in range(2, highest + 1)}
        for sequence in sequences:
            padded = BOS * (highest - 1) + sequence + EOS
            for order, counts in self.raw.items():
                counts.update(padded[position - order + 1:position + 1] for position in range(highest - 1, len(padded)))
        self.continuation = {}
        for order in range(1, highest):
            effective = Counter()
            for gram in self.raw[order + 1]:
                effective[gram[1:]] += 1
            self.continuation[order] = effective
        self.base = self.continuation[1]
        self.base_total = sum(self.base.values()) + BASE_EPSILON * len(self.support)
        self.discount_records = []
        def prepare(count_tables, origin):
            discounts_by_order, histories_by_order = {}, {}
            for order, counts in count_tables.items():
                if order == 1:
                    continue
                discounts, histogram = estimate_discounts(counts)
                discounts_by_order[order] = discounts
                totals, removed = Counter(), Counter()
                for gram, count in counts.items():
                    totals[gram[:-1]] += count
                    removed[gram[:-1]] += discount_for(count, discounts)
                histories_by_order[order] = {history: (total, removed[history]) for history, total in totals.items()}
                self.discount_records.append(dict(origen=origin, orden=order, D1=discounts[0], D2=discounts[1], D3=discounts[2], **histogram))
            return discounts_by_order, histories_by_order
        self.raw_discounts, self.raw_hist = prepare(self.raw, 'frecuencias')
        self.cont_discounts, self.cont_hist = prepare(self.continuation, 'continuaciones')
        self.lower_cached = lru_cache(maxsize=200000)(self._lower)
        self.log_cached = lru_cache(maxsize=200000)(self._log_gram)
        return self

    def _lower(self, gram):
        if len(gram) == 1:
            return (self.base.get(gram, 0) + BASE_EPSILON) / self.base_total
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.cont_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.continuation[order].get(gram, 0)
        return (count - discount_for(count, self.cont_discounts[order]) + removed * lower) / total

    def probability(self, gram):
        if len(gram) == 1:
            return self.lower_cached(gram)
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.raw_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.raw[order].get(gram, 0)
        return (count - discount_for(count, self.raw_discounts[order]) + removed * lower) / total

    def _log_gram(self, gram):
        probability = self.probability(gram)
        if not 0 < probability <= 1 + 1e-10:
            raise ArithmeticError((repr(gram), probability))
        return math.log(probability)

    def log_probability(self, sequence, order):
        assert 1 <= order <= self.max_order
        padded = BOS * (order - 1) + sequence + EOS
        grams = Counter(padded[position - order + 1:position + 1] for position in range(order - 1, len(padded)))
        return sum(count * self.log_cached(gram) for gram, count in grams.items())

    def close(self):
        self.lower_cached.cache_clear()
        self.log_cached.cache_clear()

def test_modified_kn():
    values, _ = estimate_discounts(Counter({'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 1, 'f': 2, 'g': 3, 'h': 1}))
    assert len(set(values)) == 3 and all(0 < value < bucket for bucket, value in enumerate(values, 1))
    model = ModifiedKN(4, set('abc')).fit(['ab', 'ab', 'ac', 'bc', 'abc', 'abc', 'abc'])
    assert model.base['a'] == 1 and model.raw[2][BOS + 'a'] == 6
    for order in [1, 2, 3, 4]:
        contexts = set(model.raw_hist.get(order, {})) | {'', 'zz', BOS * (order - 1)}
        for context in contexts:
            history = (BOS * (order - 1) + context)[-(order - 1):] if order > 1 else ''
            assert abs(sum(model.probability(history + symbol) for symbol in model.support) - 1) < 1e-9, (order, repr(history))
        assert math.isfinite(model.log_probability('a' + UNK, order))
    assert model.log_probability('ab', 4) > model.log_probability('cc', 4)
    model.close()
    assert tokenize('Hola HOLA', 'word') == ['hola', 'hola']
    assert tokenize('Hola', 'char')[0] == 'H'
    print('Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.')

def test_shared_counts():
    toy_texts=['Hola hola!','Hola amiga!','Adios hola!','Hola hola!','Amiga adios!']
    for level,orders in [('char',CHAR_ORDERS),('word',WORD_ORDERS)]:
        mapping=make_encoder(toy_texts,level)
        sequences=[encode_text(text,level,mapping) for text in toy_texts]
        support=set(mapping.values())
        shared_model=ModifiedKN(max(orders),support).fit(sequences)
        for order in orders:
            separate_model=ModifiedKN(order,support).fit(sequences)
            for sequence in sequences+[UNK]:
                assert abs(shared_model.log_probability(sequence,order)-separate_model.log_probability(sequence,order))<1e-12,(level,order)
            separate_model.close()
        shared_model.close()
    print('Prueba de conteos compartidos: mismos log P que entrenar cada orden por separado.')

test_modified_kn()
test_shared_counts()
timing_rows=[]

def fit_score_orders(fit_texts,fit_labels,app_texts,level,orders,phase,fold):
    started=time.perf_counter()
    mapping=make_encoder(fit_texts,level)
    encoded_fit=[encode_text(text,level,mapping) for text in fit_texts]
    encoded_app=[encode_text(text,level,mapping) for text in app_texts]
    preprocessing=time.perf_counter()-started
    matrices={order:np.zeros((len(app_texts),6)) for order in orders}
    fit_seconds=0.0
    score_seconds={order:0.0 for order in orders}
    for class_index in range(6):
        fit_started=time.perf_counter()
        model=ModifiedKN(max(orders),set(mapping.values())).fit(sequence for sequence,target in zip(encoded_fit,fit_labels) if target==class_index)
        fit_seconds+=time.perf_counter()-fit_started
        for order in orders:
            score_started=time.perf_counter()
            matrices[order][:,class_index]=[model.log_probability(sequence,order) for sequence in encoded_app]
            score_seconds[order]+=time.perf_counter()-score_started
        model.close()
        del model
        gc.collect()
    for order in orders:
        timing_rows.append(dict(fase=phase,pliegue=fold,nivel=level,orden=order,ajuste_compartido_s=preprocessing+fit_seconds,puntuacion_s=score_seconds[order],tiempo_amortizado_s=score_seconds[order]+(preprocessing+fit_seconds)/len(orders)))
    print(phase,fold,level,'ordenes',orders,'segundos',round(time.perf_counter()-started,2),flush=True)
    return matrices


Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.
Prueba de conteos compartidos: mismos log P que entrenar cada orden por separado.


## Metodo: PPM-C de caracteres, orden 4, con exclusion

In [4]:
PPM_ORDER=4
LOG2=math.log(2)

class PPMC:
    # PPM-C estatico de caracteres: escape C, exclusion de simbolos vistos en ordenes superiores, orden -1 uniforme.
    def __init__(self,alphabet,max_order=PPM_ORDER):
        self.alphabet=frozenset(alphabet)|{EOS,UNK}
        self.max_order=max_order
    def fit(self,sequences):
        grams={order:Counter() for order in range(self.max_order+1)}
        for sequence in sequences:
            text=sequence+EOS
            for order,counts in grams.items():
                counts.update(text[position-order:position+1] for position in range(order,len(text)))
        self.tables={}
        for order,counts in grams.items():
            for gram,count in counts.items():
                self.tables.setdefault(gram[:-1],{})[gram[-1]]=count
        self.totals={context:(sum(table.values()),len(table)) for context,table in self.tables.items()}
        self.log2_cached=lru_cache(maxsize=1000000)(self._log2_probability)
        return self
    def probability(self,history,symbol):
        excluded=()
        probability=1.0
        for order in range(min(self.max_order,len(history)),-1,-1):
            context=history[len(history)-order:]
            table=self.tables.get(context)
            if table is None:
                continue
            total,distinct=self.totals[context]
            if excluded:
                total-=sum(table.get(previous,0) for previous in excluded)
                distinct-=sum(previous in table for previous in excluded)
            if distinct==0:
                continue
            if symbol in table and symbol not in excluded:
                return probability*table[symbol]/(total+distinct)
            probability*=distinct/(total+distinct)
            excluded=tuple(set(excluded)|set(table))
        remaining=len(self.alphabet-set(excluded))
        return probability/remaining
    def _log2_probability(self,history,symbol):
        probability=self.probability(history,symbol)
        if not 0<probability<=1+1e-12:
            raise ArithmeticError((repr(history),repr(symbol),probability))
        return math.log(probability)/LOG2
    def log2_likelihood(self,sequence):
        text=sequence+EOS
        return sum(self.log2_cached(text[max(0,position-self.max_order):position],text[position]) for position in range(len(text)))
    def close(self):
        self.log2_cached.cache_clear()

def map_alphabet(text,alphabet):
    return ''.join(character if character in alphabet else UNK for character in text)

def test_ppmc():
    model=PPMC(set('abrcd')).fit(['abracadabra'])
    for history in ['', 'a', 'ab', 'abr', 'abra', 'zzzz', 'cada', 'xa']:
        total=sum(model.probability(history,symbol) for symbol in model.alphabet)
        assert abs(total-1)<1e-12,(history,total)
    # Contexto 'abra' -> 'c' (visto 1 vez de 2 apariciones del contexto con 2 simbolos distintos {c, EOS}): 1/(2+2).
    assert abs(model.probability('abra','c')-1/4)<1e-12
    # Con exclusion, 'b' tras 'abra' escapa de orden 4 (2/4) y en orden 3 'bra' ya solo tiene {c, EOS} excluidos -> se salta, etc.
    no_exclusion=0.5*(model.tables['a']['b']/(model.totals['a'][0]+model.totals['a'][1]))
    assert model.probability('abra','b')!=no_exclusion
    assert math.isfinite(model.log2_likelihood('abc'+UNK))
    model.close()
    print('Prueba PPM-C: distribuciones suman 1 en contextos vistos, no vistos y de inicio; escape C y exclusion verificados.')

test_ppmc()

def fit_score_method(fit_texts,fit_labels,app_texts):
    # score = log2 P(texto + fin | clase) en bits + log2 prior.
    alphabet=set().union(*map(set,fit_texts))
    app_mapped=[map_alphabet(text,alphabet) for text in app_texts]
    prior=np.log2(np.bincount(fit_labels,minlength=6)/len(fit_labels))
    scores=np.zeros((len(app_texts),6))
    for class_index in range(6):
        model=PPMC(alphabet).fit(fit_texts[index] for index in np.flatnonzero(fit_labels==class_index))
        scores[:,class_index]=[prior[class_index]+model.log2_likelihood(text) for text in app_mapped]
        model.close()
        del model
        gc.collect()
    return scores,dict(alfabeto=len(alphabet)+2,tasa_desconocidos=float(np.mean([character==UNK for text in app_mapped for character in text])))


Prueba PPM-C: distribuciones suman 1 en contextos vistos, no vistos y de inicio; escape C y exclusion verificados.


## Validacion cruzada: referencia A recalculada y metodo

In [5]:
def macro_f1(y_true,y_pred):
    return f1_score(y_true,y_pred,labels=np.arange(6),average='macro',zero_division=0)

cv_started=time.perf_counter()
oof={key:np.zeros((len(y_development),6)) for key in ['REF',METHOD]}
fold_rows=[]
for fold_number,(fit_idx,app_idx) in enumerate(folds,1):
    started=time.perf_counter()
    fit_texts,app_texts=texts_development[fit_idx],texts_development[app_idx]
    fit_labels=y_development[fit_idx]
    assert not {text.lower() for text in fit_texts}&{text.lower() for text in app_texts}
    characters=fit_score_orders(fit_texts,fit_labels,app_texts,'char',[REF_CHAR_ORDER],'CV',fold_number)[REF_CHAR_ORDER]
    words=fit_score_orders(fit_texts,fit_labels,app_texts,'word',[REF_WORD_ORDER],'CV',fold_number)[REF_WORD_ORDER]
    prior=np.log(np.bincount(fit_labels,minlength=6)/len(fit_labels))
    oof['REF'][app_idx]=REF_WEIGHT*characters+(1-REF_WEIGHT)*words+prior
    reference_seconds=time.perf_counter()-started
    method_started=time.perf_counter()
    oof[METHOD][app_idx],details=fit_score_method(fit_texts,fit_labels,app_texts)
    method_seconds=time.perf_counter()-method_started
    y_app=y_development[app_idx]
    fold_rows.append(dict(pliegue=fold_number,f1_referencia=macro_f1(y_app,oof['REF'][app_idx].argmax(axis=1)),f1_metodo=macro_f1(y_app,oof[METHOD][app_idx].argmax(axis=1)),segundos_referencia=reference_seconds,segundos_metodo=method_seconds,**details))
    print(f"pliegue {fold_number}: F1 A {fold_rows[-1]['f1_referencia']:.6f} | F1 {METHOD} {fold_rows[-1]['f1_metodo']:.6f} | {METHOD} {method_seconds:.1f} s",flush=True)
    del characters,words
    gc.collect()
folds_table=pd.DataFrame(fold_rows)
folds_table['diferencia_con_A']=folds_table.f1_metodo-folds_table.f1_referencia
reference_oof=macro_f1(y_development,oof['REF'].argmax(axis=1))
reference_gap=np.abs(folds_table.f1_referencia.to_numpy()-REF_FOLD_F1)
print('Referencia A recalculada por pliegue:',folds_table.f1_referencia.round(6).tolist(),'| OOF',round(reference_oof,6))
print('Max |diferencia| con la referencia medida por pliegue:',float(reference_gap.max()),'| OOF:',abs(reference_oof-REF_OOF_F1))
if reference_gap.max()>REF_TOLERANCE or abs(reference_oof-REF_OOF_F1)>REF_TOLERANCE:
    raise RuntimeError('DETENER: la referencia A recalculada no coincide con la medida (tolerancia 1e-6).')
print('REFERENCIA A VERIFICADA (tolerancia 1e-6).')
differences=folds_table.diferencia_con_A.to_numpy()
results=pd.DataFrame([
    dict(modelo='A (KN car 4 + KN pal 3, lambda 0,75)',f1_macro_oof=reference_oof,f1_pliegues_media=float(folds_table.f1_referencia.mean()),f1_pliegues_std=float(folds_table.f1_referencia.std(ddof=1)),diferencia_pareada_media=0.0,diferencia_pareada_std=0.0),
    dict(modelo=METHOD_LABEL,f1_macro_oof=macro_f1(y_development,oof[METHOD].argmax(axis=1)),f1_pliegues_media=float(folds_table.f1_metodo.mean()),f1_pliegues_std=float(folds_table.f1_metodo.std(ddof=1)),diferencia_pareada_media=float(differences.mean()),diferencia_pareada_std=float(differences.std(ddof=1))),
])
print('TABLA: F1 macro OOF sin calibrar, F1 por pliegue y diferencia pareada contra A (ddof=1)')
print(results.to_string(index=False))
display(folds_table)
print('CV segundos:',round(time.perf_counter()-cv_started,1))


CV 1 char ordenes [4] segundos 6.79


CV 1 word ordenes [3] segundos 5.55


pliegue 1: F1 A 0.440843 | F1 PPM 0.421778 | PPM 11.2 s


CV 2 char ordenes [4] segundos 7.44


CV 2 word ordenes [3] segundos 5.44


...

CV 5 word ordenes [3] segundos 5.47


pliegue 5: F1 A 0.439014 | F1 PPM 0.426628 | PPM 10.8 s


Referencia A recalculada por pliegue: [0.440843, 0.432485, 0.440697, 0.447354, 0.439014] | OOF 0.440081
Max |diferencia| con la referencia medida por pliegue: 4.232924309954811e-07 | OOF: 2.786557278255941e-07
REFERENCIA A VERIFICADA (tolerancia 1e-6).
TABLA: F1 macro OOF sin calibrar, F1 por pliegue y diferencia pareada contra A (ddof=1)
                                     modelo  f1_macro_oof  f1_pliegues_media  f1_pliegues_std  diferencia_pareada_media  diferencia_pareada_std
       A (KN car 4 + KN pal 3, lambda 0,75)      0.440081           0.440079         0.005308                  0.000000                0.000000
PPM-C de caracteres, orden 4, con exclusion      0.421730           0.421750         0.002934                 -0.018329                0.005625


,pliegue,f1_referencia,f1_metodo,segundos_referencia,segundos_metodo,alfabeto,tasa_desconocidos,diferencia_con_A
0,1,0.440843,0.421778,12.370534,11.247051,98,4.853926e-07,-0.019065
1,2,0.432485,0.419215,12.901916,11.176248,99,0.000000e+00,-0.013271
2,3,0.440697,0.419726,12.242342,10.383905,99,0.000000e+00,-0.020971
3,4,0.447354,0.421401,12.034945,11.412130,99,0.000000e+00,-0.025953
4,5,0.439014,0.426628,12.279285,10.821382,99,0.000000e+00,-0.012385


CV segundos: 117.0


## Decision

In [6]:
method_row=results.iloc[1]
PASA=bool(method_row.diferencia_pareada_media>0 and method_row.diferencia_pareada_media>method_row.diferencia_pareada_std)
print('Diferencia pareada media:',f'{method_row.diferencia_pareada_media:+.6f}','| desviacion estandar (ddof=1):',f'{method_row.diferencia_pareada_std:.6f}')
print('Regla fijada: media de diferencia pareada >0 y > su desviacion estandar muestral.')
print('PASA' if PASA else 'NO PASA')
if not PASA:
    print(f'NO PASA: holdout y CSV se omiten explicitamente; no se creara {SUB_PATH.name}.')


Diferencia pareada media: -0.018329 | desviacion estandar (ddof=1): 0.005625
Regla fijada: media de diferencia pareada >0 y > su desviacion estandar muestral.
NO PASA
NO PASA: holdout y CSV se omiten explicitamente; no se creara age_submission_v8_ppm.csv.


## Holdout (solo si PASA)

In [7]:
if PASA:
    assert 'holdout_f1' not in globals(),'Holdout ya evaluado; no reutilizar.'
    started=time.perf_counter()
    texts_holdout=train.normalized.iloc[idx_holdout].to_numpy()
    assert not {text.lower() for text in texts_development}&{text.lower() for text in texts_holdout}
    holdout_scores,_=fit_score_method(texts_development,y_development,texts_holdout)
    holdout_pred=holdout_scores.argmax(axis=1)
    y_holdout=y_all[idx_holdout]
    holdout_f1=macro_f1(y_holdout,holdout_pred)
    print('F1 macro holdout v7_lower:',holdout_f1)
    display(pd.DataFrame(classification_report(y_holdout,holdout_pred,labels=np.arange(6),target_names=CLASSES,output_dict=True,zero_division=0)).T.loc[CLASSES,['precision','recall','f1-score','support']])
    print('Holdout segundos:',round(time.perf_counter()-started,1),'; evaluacion unica. No comparar directamente con el holdout de v6 ni con Kaggle.')
else:
    print('HOLDOUT OMITIDO: NO PASA; no se entrena ni evalua en holdout.')


HOLDOUT OMITIDO: NO PASA; no se entrena ni evalua en holdout.


## CSV (solo si PASA)

In [8]:
if PASA:
    verify_csv(DATA_DIR/'age_test.csv',['id','text'],';',True)
    test=pd.read_csv(DATA_DIR/'age_test.csv',sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
    assert list(test.columns)==['id','text'] and test.id.is_unique and test.id.ne('').all()
    test_texts=test.text.fillna('').str.strip().map(html.unescape).map(normalize_text).to_numpy()
    test_scores,_=fit_score_method(train.normalized.to_numpy(),y_all,test_texts)
    submission=pd.DataFrame({'id':test.id,'age_range':CLASSES[test_scores.argmax(axis=1)]})
    assert list(submission.columns)==reference_columns==['id','age_range']
    assert len(submission)==len(test)==112892 and submission.id.is_unique and submission.id.tolist()==test.id.tolist()
    assert submission.age_range.isin(CLASSES).all()
    with SUB_PATH.open('x',encoding='utf-8',newline='') as handle:
        submission.to_csv(handle,index=False,sep=',')
    print('CSV VERIFICADO:',SUB_PATH.name,len(submission),'filas; coma y sin BOM.')
else:
    assert not SUB_PATH.exists()
    print('CSV OMITIDO: NO PASA; no se carga el test para predecir ni se genera salida.')


CSV OMITIDO: NO PASA; no se carga el test para predecir ni se genera salida.


## Matriz de confusion OOF del metodo

In [9]:
predictions=oof[METHOD].argmax(axis=1)
matrix=confusion_matrix(y_development,predictions,labels=np.arange(6))
print('Matriz de confusion OOF del metodo:',METHOD_LABEL,'| F1 macro OOF',round(float(results.iloc[1].f1_macro_oof),6))
display(pd.DataFrame(matrix,index=[f'real {name}' for name in CLASSES],columns=[f'pred {name}' for name in CLASSES]))
recall=matrix.diagonal()/matrix.sum(axis=1)
predicted_share=matrix.sum(axis=0)/matrix.sum()
off_diagonal=[(matrix[i,j],CLASSES[i],CLASSES[j]) for i in range(6) for j in range(6) if i!=j]
adjacent_share=sum(matrix[i,j] for i in range(6) for j in range(6) if abs(i-j)==1)/max(1,matrix.sum()-matrix.trace())
print('Recall por clase:',{str(name):round(float(value),3) for name,value in zip(CLASSES,recall)})
print('Proporcion de predicciones por clase:',{str(name):round(float(value),3) for name,value in zip(CLASSES,predicted_share)})
print('Mayores confusiones:',[(f'{real}->{pred}',int(count)) for count,real,pred in sorted(off_diagonal,reverse=True)[:4]])
print(f'HIPOTESIS (no verificada): {adjacent_share:.1%} de los errores caen en rangos adyacentes; si las predicciones se concentran en pocas clases, el modelo estaria dominado por diferencias de longitud o de frecuencia global de simbolos mas que por estilo. Requiere analisis cualitativo antes de afirmarlo.')


Matriz de confusion OOF del metodo: PPM-C de caracteres, orden 4, con exclusion | F1 macro OOF 0.42173


,pred 13-17,pred 18-24,pred 25-34,pred 35-49,pred 50-64,pred 65-xx
real 13-17,10889,5346,2681,1513,1355,1008
real 18-24,5023,8201,3932,2222,1715,1459
real 25-34,2885,4252,7063,3298,2682,2225
real 35-49,1528,2355,3546,7330,3934,3475
real 50-64,898,1210,1965,2786,10943,4221
real 65-xx,763,1113,1337,2142,3771,11921


Recall por clase: {'13-17': 0.478, '18-24': 0.364, '25-34': 0.315, '35-49': 0.331, '50-64': 0.497, '65-xx': 0.566}
Proporcion de predicciones por clase: {'13-17': 0.165, '18-24': 0.169, '25-34': 0.154, '35-49': 0.145, '50-64': 0.183, '65-xx': 0.183}
Mayores confusiones: [('13-17->18-24', 5346), ('18-24->13-17', 5023), ('25-34->18-24', 4252), ('50-64->65-xx', 4221)]
HIPOTESIS (no verificada): 52.3% de los errores caen en rangos adyacentes; si las predicciones se concentran en pocas clases, el modelo estaria dominado por diferencias de longitud o de frecuencia global de simbolos mas que por estilo. Requiere analisis cualitativo antes de afirmarlo.


## Resumen

In [10]:
method_oof=float(results.iloc[1].f1_macro_oof)
summary_text=f'''## Resumen final: {METHOD_LABEL}

Resultado: **{'PASA' if PASA else 'NO PASA'}**. F1 macro OOF sin calibrar = {method_oof:.6f} frente a A recalculada = {reference_oof:.6f}; diferencia pareada media {results.iloc[1].diferencia_pareada_media:+.6f} (desviacion estandar {results.iloc[1].diferencia_pareada_std:.6f}, ddof=1). {'Holdout evaluado una sola vez: F1 = ' + format(holdout_f1, '.6f') + '; CSV generado con escritura exclusiva.' if PASA else 'Holdout y CSV omitidos explicitamente por la regla.'}

**Optimismo de seleccion:** en esta ronda se prueban dos metodos (Naive Bayes y PPM-C) contra una misma referencia, lo que introduce un optimismo de seleccion minimo. Ambos son modelos de lenguaje por clase de la misma familia generativa que v5-v7. La regla es una heuristica, no una prueba formal.

**Comparabilidad:** la particion v7_lower no es directamente comparable con el holdout de v6; los scores publicos de Kaggle no son comparables con holdout. Referencias historicas sobre particiones distintas: v5 calibrado 0,43847, v6 sin calibrar 0,44106 y v6 calibrado 0,44353.

NPZ v7_lower reutilizado sin regenerar, con hashes de archivo, train y grupos verificados y cero fuga en minusculas. Sin calibracion de sesgos, sin commit, push ni envios a Kaggle. Entorno .venv-v6, kernel tweetage-v6-kn.
'''
display(Markdown(summary_text))
assert INPUT_HASHES=={name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in INPUT_HASHES}
assert PROTECTED_HASHES=={name:hashlib.sha256((OUT_DIR/name).read_bytes()).hexdigest() for name in PROTECTED_HASHES}
print('HASHES DE TODOS LOS ARCHIVOS PREEXISTENTES PROTEGIDOS: SIN CAMBIOS.')


## Resumen final: PPM-C de caracteres, orden 4, con exclusion

Resultado: **NO PASA**. F1 macro OOF sin calibrar = 0.421730 frente a A recalculada = 0.440081; diferencia pareada media -0.018329 (desviacion estandar 0.005625, ddof=1). Holdout y CSV omitidos explicitamente por la regla.

**Optimismo de seleccion:** en esta ronda se prueban dos metodos (Naive Bayes y PPM-C) contra una misma referencia, lo que introduce un optimismo de seleccion minimo. Ambos son modelos de lenguaje por clase de la misma familia generativa que v5-v7. La regla es una heuristica, no una prueba formal.

**Comparabilidad:** la particion v7_lower no es directamente comparable con el holdout de v6; los scores publicos de Kaggle no son comparables con holdout. Referencias historicas sobre particiones distintas: v5 calibrado 0,43847, v6 sin calibrar 0,44106 y v6 calibrado 0,44353.

NPZ v7_lower reutilizado sin regenerar, con hashes de archivo, train y grupos verificados y cero fuga en minusculas. Sin calibracion de sesgos, sin commit, push ni envios a Kaggle. Entorno .venv-v6, kernel tweetage-v6-kn.


HASHES DE TODOS LOS ARCHIVOS PREEXISTENTES PROTEGIDOS: SIN CAMBIOS.
